In [ ]:
# ==========================================================
# 1. KÜTÜPHANELER
# ==========================================================

import cv2
import os
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import load_model

print("TensorFlow :", tf.__version__)
print("OpenCV :", cv2.__version__)

TensorFlow : 2.20.0
OpenCV : 4.13.0


In [ ]:
# ==========================================================
# 2. DRIVE BAĞLANTI
# ==========================================================

from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ==========================================================
# PATHS
# ==========================================================

MODEL_PATH = "/content/drive/MyDrive/MyDrive/Driver_Project/models/best_model.keras"

VIDEO_PATH = "/content/drive/MyDrive/MyDrive/Driver_Project/videos/run1b_2018-05-29-14-02-47.kinect_color.mp4"

IMG_SIZE = (224,224)

CLASS_NAMES = [
    "Safe Driving",
    "Text Right",
    "Talk Right",
    "Text Left",
    "Talk Left",
    "Adjust Radio",
    "Drink",
    "Reach Behind",
    "Hair & Makeup",
    "Talk Passenger"
]

print("Hazır.")

Hazır.


In [ ]:
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras import layers

preprocessing = layers.Lambda(preprocess_input)

model = load_model(
    MODEL_PATH,
    custom_objects={
        "preprocess_input": preprocess_input,
        "preprocessing": preprocessing
    },
    safe_mode=False
)

print("✅ Model yüklendi.")

✅ Model yüklendi.


In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    print("❌ Video açılamadı.")
else:
    print("✅ Video açıldı.")

✅ Video açıldı.


In [ ]:
fps = cap.get(cv2.CAP_PROP_FPS)
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

duration = frame_count / fps

print(f"FPS           : {fps:.2f}")
print(f"Frame Sayısı  : {frame_count}")
print(f"Süre          : {duration:.2f} saniye")

FPS           : 15.00
Frame Sayısı  : 19072
Süre          : 1271.47 saniye


In [ ]:
FRAME_INTERVAL = int(fps * 2)

print("Her", FRAME_INTERVAL, "frame'de bir analiz yapılacak.")

Her 30 frame'de bir analiz yapılacak.


In [ ]:
CLASS_NAMES = [
    "Safe Driving",      # c0
    "Text Right",        # c1
    "Talk Right",        # c2
    "Text Left",         # c3
    "Talk Left",         # c4
    "Adjust Radio",      # c5
    "Drink",             # c6
    "Reach Behind",      # c7
    "Hair & Makeup",     # c8
    "Talk Passenger"     # c9
]

In [ ]:
frame_id = 0

while True:

    ret, frame = cap.read()

    if not ret:
        break

    if frame_id % FRAME_INTERVAL == 0:

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        img = cv2.resize(rgb, (224,224))
        img = np.expand_dims(img, axis=0)

        prediction = model.predict(img, verbose=0)

        pred_class = np.argmax(prediction)
        confidence = np.max(prediction)

        second = frame_id / fps

        print(
            f"{second:7.1f} sn | "
            f"{CLASS_NAMES[pred_class]:15} | "
            f"{confidence:.3f}"
        )

    frame_id += 1

cap.release()

    0.0 sn | Hair & Makeup   | 0.552
    2.0 sn | Hair & Makeup   | 0.518
    4.0 sn | Hair & Makeup   | 0.784
    6.0 sn | Reach Behind    | 0.382
    8.0 sn | Hair & Makeup   | 0.618
   10.0 sn | Text Left       | 0.802
   12.0 sn | Text Left       | 0.907
   14.0 sn | Talk Passenger  | 0.442
   16.0 sn | Hair & Makeup   | 0.365
   18.0 sn | Talk Passenger  | 0.556
   20.0 sn | Safe Driving    | 0.584
   22.0 sn | Safe Driving    | 0.545
   24.0 sn | Text Left       | 0.509
   26.0 sn | Safe Driving    | 0.678
   28.0 sn | Safe Driving    | 0.907
   30.0 sn | Safe Driving    | 0.699
   32.0 sn | Safe Driving    | 0.660
   34.0 sn | Safe Driving    | 0.463
   36.0 sn | Safe Driving    | 0.550
   38.0 sn | Safe Driving    | 0.451
   40.0 sn | Safe Driving    | 0.495
   42.0 sn | Safe Driving    | 0.665
   44.0 sn | Safe Driving    | 0.657
   46.0 sn | Safe Driving    | 0.493
   48.0 sn | Hair & Makeup   | 0.551
   50.0 sn | Talk Passenger  | 0.264
   52.0 sn | Hair & Makeup   | 0.536
 

KeyboardInterrupt: 

In [ ]:
import cv2
import numpy as np

OUTPUT_PATH = "/content/drive/MyDrive/MyDrive/Driver_Project/results/result.mp4"

In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')

writer = cv2.VideoWriter(
    OUTPUT_PATH,
    fourcc,
    fps,
    (width, height)
)

print("Video hazır.")

Video hazır.


In [ ]:
frame_id = 0

current_label = "..."
current_conf = 0

cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

while True:

    ret, frame = cap.read()

    if not ret:
        break

    if frame_id % FRAME_INTERVAL == 0:

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        img = cv2.resize(rgb, (224,224))
        img = np.expand_dims(img, axis=0)

        prediction = model.predict(img, verbose=0)

        pred = np.argmax(prediction)

        current_label = CLASS_NAMES[pred]
        current_conf = float(np.max(prediction))

    cv2.putText(
        frame,
        f"Action: {current_label}",
        (20,40),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0,255,0),
        2
    )

    cv2.putText(
        frame,
        f"Confidence: {current_conf:.2f}",
        (20,80),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0,255,0),
        2
    )

    writer.write(frame)

    frame_id += 1

cap.release()
writer.release()

print("✅ Video oluşturuldu.")
print(OUTPUT_PATH)

✅ Video oluşturuldu.
/content/drive/MyDrive/MyDrive/Driver_Project/results/result.mp4
